In [1]:
import pandas as pd

In [2]:
#mfp_df = pd.read_csv('mfp_season_df_ifpa.csv')
series_id = '2149'
mfp_df = pd.read_csv('mfp_league_'+series_id+'_standings.csv')

In [3]:
# convert to csv with player name, week number, score

In [4]:
mfp_season = mfp_df.loc[:,mfp_df.columns[0:11]]

In [5]:
mfp_season.columns.values[0] = 'name'
mfp_season.head()

,name,77415,77416,77417,77418,77419,77420,avg_score,weeks_played,avg_att,proj_att
0,Greg Dunlap,27.0,27.0,31.0,31.0,21.0,23.0,26.666667,6,1.000000,10
1,Matthew Talley,27.0,26.0,25.0,27.0,27.0,19.0,25.166667,6,1.000000,10
2,Philip Ryder,19.0,27.0,23.0,27.0,19.0,31.0,24.333333,6,1.000000,10
3,John Tracey,27.0,33.0,27.0,NaN,31.0,25.0,28.600000,5,0.833333,8
4,BeeOtch Westrum,31.0,14.0,15.0,27.0,25.0,29.0,23.500000,6,1.000000,10


In [6]:
week_ids = ['28493','28494','28495','28496','28497','28498','28499','28500','28501','28502']

In [7]:
mfp_season.head()

,name,77415,77416,77417,77418,77419,77420,avg_score,weeks_played,avg_att,proj_att
0,Greg Dunlap,27.0,27.0,31.0,31.0,21.0,23.0,26.666667,6,1.000000,10
1,Matthew Talley,27.0,26.0,25.0,27.0,27.0,19.0,25.166667,6,1.000000,10
2,Philip Ryder,19.0,27.0,23.0,27.0,19.0,31.0,24.333333,6,1.000000,10
3,John Tracey,27.0,33.0,27.0,NaN,31.0,25.0,28.600000,5,0.833333,8
4,BeeOtch Westrum,31.0,14.0,15.0,27.0,25.0,29.0,23.500000,6,1.000000,10


In [8]:
mfp_season.fillna(0,inplace=True)

In [9]:
def find_weekly_scores(row):
    for i in range(len(week_ids)+1):
        if len(row[1:i+1]) < 5:
            row[str(i)] = sum(row[1:i+1])
        else:
            row[str(i)] = sum(sorted(row[1:i+1],reverse=True)[:5])
    
    return row

In [10]:
mfp_season = mfp_season.apply(find_weekly_scores,axis=1)

In [11]:
mfp_season = mfp_season.loc[:,['name','1','2','3','4','5','6','7','8','9','10']]

In [12]:
mfp_season = mfp_season.melt(id_vars = 'name',value_vars = ['1','2','3','4','5','6','7','8','9','10'], var_name = 'week')

In [13]:
mfp_season.tail()

,name,week,value
765,Lisa Barrrera,10,25.166667
766,Adam Lam,10,23.166667
767,Sarah Twigg,10,19.166667
768,Victor Group,10,19.166667
769,Carlene Malack,10,15.166667


In [14]:
def find_last_value(row):
#     if row.week == '1':
#         row['last_value'] = 0
#     else:
#         temp = mfp_season[mfp_season.name == row['name']]
#         row['last_value'] = int(temp[temp.week == str(int(row['week'])-1)].value)
    try:
        temp = mfp_season[mfp_season.name == row['name']]
        row['last_value'] = int(temp[temp.week == str(int(row['week']) -1)].value)
    except:
        row['last_value'] = 0
    return row

In [15]:
mfp_season = mfp_season.apply(find_last_value,axis=1)

In [17]:
def find_rank(row):
    temp = mfp_season[mfp_season.week == row['week']]
    temp['rank'] = temp.value.rank(method='min',ascending=False)
    row['rank'] = temp.loc[temp.name == row['name']]['rank']
    return row

In [18]:
mfp_season = mfp_season.apply(find_rank, axis=1)

In [19]:
mfp_season.sort_values('value',ascending=False, inplace=True)

In [20]:
mfp_season.to_csv('mfp_season_'+series_id+'_by_week.csv',float_format='%.df')

In [21]:
mfp_season

,name,week,value,last_value,rank
619,John Tracey,9,146.6,146,"619 1.0 Name: rank, dtype: float64"
542,John Tracey,8,146.6,146,"542 1.0 Name: rank, dtype: float64"
465,John Tracey,7,146.6,143,"465 1.0 Name: rank, dtype: float64"
696,John Tracey,10,146.6,146,"696 1.0 Name: rank, dtype: float64"
388,John Tracey,6,143.0,118,"388 1.0 Name: rank, dtype: float64"
...,...,...,...,...,...
58,Dale Green,1,0.0,0,"58 46.0 Name: rank, dtype: float64"
59,Mike Nedresky,1,0.0,0,"59 46.0 Name: rank, dtype: float64"
60,Glenn Howard-Erevia,1,0.0,0,"60 46.0 Name: rank, dtype: float64"
61,Greg Friedman,1,0.0,0,"61 46.0 Name: rank, dtype: float64"


https://observablehq.com/@willnetsky/bar-chart-race-mfp-winter-2019